# RAG Document Question Answering System

**Name**: Sahil Yadav  
**Week 7 - Celebal Technologies Internship**

## What is RAG?

RAG stands for Retrieval-Augmented Generation. The basic idea is:
1. Take a document (like a PDF)
2. Break it into small chunks
3. Convert each chunk into a vector embedding
4. Store these vectors in a database
5. When someone asks a question, find the most relevant chunks
6. Feed those chunks to an LLM to generate an answer

This way the LLM answers based on YOUR documents, not just its training data.

## Tools Used
- **Cohere** - for creating embeddings and generating answers
- **Pinecone** - vector database for storing/searching embeddings
- **PyMuPDF** - for reading PDF files
- **Streamlit** - for the web interface

## Step 0: Install Dependencies

First let's install everything we need.

In [ ]:
!pip install cohere==5.9.0 pinecone-client==5.0.0 PyMuPDF==1.24.14 streamlit==1.33.0 sentence-transformers==3.3.1

## Step 1: PDF Text Extraction

The first step is to read the PDF and get all the text out of it. We use PyMuPDF (imported as `fitz`) for this. It's pretty straightforward - just loop through each page and grab the text.

In [ ]:
import fitz  # PyMuPDF - weird name but this is how you import it

def extract_text_from_pdf(pdf_path):
    """Read a PDF file and return all the text"""
    text = ""
    with fitz.open(pdf_path) as pdf:
        print(f"PDF has {pdf.page_count} pages")
        for page_num in range(pdf.page_count):
            page = pdf.load_page(page_num)
            text += page.get_text("text")
    return text

# test with a sample PDF (change this path to your PDF)
# pdf_text = extract_text_from_pdf("your_document.pdf")
# print(f"Extracted {len(pdf_text)} characters")
# print(pdf_text[:500])  # preview first 500 chars

print("PDF extraction function ready!")

## Step 2: Text Chunking

We can't just throw the entire document at the LLM - it might be too long, and we want to find *specific* relevant parts. So we split the text into smaller chunks.

I split by sentences first (using ". " as delimiter) and then group them into chunks of about 1000 characters. This way we don't accidentally cut a sentence in half.

In [ ]:
def split_into_chunks(text, chunk_size=1000):
    """
    Split text into chunks of roughly chunk_size characters.
    We split by sentences so we don't break mid-sentence.
    """
    sentences = text.split(". ")
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
        else:
            current_chunk += sentence + ". "
    
    # don't forget the last chunk!
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks

# demo with some sample text
sample_text = "Machine learning is a subset of artificial intelligence. It allows computers to learn from data. Deep learning uses neural networks with many layers. Natural language processing deals with text data. RAG combines retrieval with generation. This makes LLMs more accurate for specific documents."

demo_chunks = split_into_chunks(sample_text, chunk_size=100)
print(f"Split into {len(demo_chunks)} chunks:")
for i, chunk in enumerate(demo_chunks):
    print(f"\nChunk {i+1} ({len(chunk)} chars): {chunk}")

## Step 3: Creating Embeddings with Cohere

Now we need to convert our text chunks into vectors (embeddings). An embedding is basically a list of numbers that represents the "meaning" of the text. Similar texts will have similar embeddings.

We use Cohere's `embed-english-v3.0` model which gives us 1024-dimensional vectors.

**Important**: When embedding documents for storage, use `input_type="search_document"`. When embedding a query, use `input_type="search_query"`. This helps Cohere optimize the embeddings for search.

In [ ]:
import cohere

# you need to put your actual API key here
COHERE_API_KEY = "your-cohere-api-key"  # replace with your key

co = cohere.Client(COHERE_API_KEY)

def embed_texts(texts, input_type="search_document"):
    """
    Create embeddings for a list of texts using Cohere.
    input_type should be 'search_document' for indexing and 'search_query' for queries.
    """
    # cohere has a batch limit so we process in groups of 96
    all_embeddings = []
    batch_size = 96
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = co.embed(
            texts=batch,
            model="embed-english-v3.0",
            input_type=input_type
        )
        all_embeddings.extend(response.embeddings)
    
    return all_embeddings

# demo - embed our sample chunks
# (uncomment after adding your API key)
# embeddings = embed_texts(demo_chunks)
# print(f"Created {len(embeddings)} embeddings")
# print(f"Each embedding has {len(embeddings[0])} dimensions")

print("Embedding function ready!")

## Step 4: Storing Vectors in Pinecone

Pinecone is a vector database - it stores our embeddings and lets us quickly find the most similar ones to a query. We create a serverless index (free tier works fine for this project).

Key settings:
- **dimension=1024** because Cohere embed-v3 produces 1024-dim vectors
- **metric="cosine"** for measuring similarity between vectors
- **ServerlessSpec** with AWS us-east-1 (free tier region)

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import uuid
import time

PINECONE_API_KEY = "your-pinecone-api-key"  # replace with your key

def create_pinecone_index(api_key, index_name=None):
    """
    Create a Pinecone index for storing our vectors.
    Returns the index object.
    """
    pc = Pinecone(api_key=api_key)
    
    # generate a unique name if not provided
    if index_name is None:
        index_name = "rag-doc-qa-" + str(uuid.uuid4())[:8]
    
    # create the index
    pc.create_index(
        name=index_name,
        dimension=1024,  # cohere embed-v3 dimension
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    
    # wait for it to be ready
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    
    print(f"Index '{index_name}' is ready!")
    return pc.Index(index_name), index_name


def upsert_vectors(index, chunks, embeddings):
    """
    Upload vectors to Pinecone with the text stored as metadata.
    """
    vectors = []
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        vectors.append({
            "id": str(i),
            "values": embedding,
            "metadata": {"text": chunk}
        })
    
    # upsert in batches of 100
    batch_size = 100
    for i in range(0, len(vectors), batch_size):
        batch = vectors[i:i + batch_size]
        index.upsert(vectors=batch)
    
    time.sleep(2)  # let pinecone process
    print(f"Uploaded {len(vectors)} vectors to Pinecone")

print("Pinecone functions ready!")

## Step 5: Retrieval - Finding Relevant Chunks

When a user asks a question, we:
1. Embed the question using Cohere (with `input_type="search_query"`)
2. Search Pinecone for the most similar document chunks
3. Use Cohere's **rerank** to improve the results

Reranking is important because initial vector search gives decent results, but reranking with a dedicated model makes them much better. We first retrieve 10 candidates, then rerank to keep the top 3.

In [ ]:
def retrieve_relevant_chunks(query, index, co_client, top_k=10, rerank_top_k=3):
    """
    Find the most relevant document chunks for a given query.
    Uses embedding search + reranking for better accuracy.
    """
    # embed the query
    query_embedding = co_client.embed(
        texts=[query],
        model="embed-english-v3.0",
        input_type="search_query"
    ).embeddings[0]
    
    # search pinecone
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )
    
    # extract the text from results
    retrieved_texts = [match['metadata']['text'] for match in results['matches']]
    
    # rerank for better results
    if retrieved_texts:
        reranked = co_client.rerank(
            query=query,
            documents=retrieved_texts,
            top_n=rerank_top_k,
            model="rerank-english-v3.0"
        )
        
        final_chunks = [retrieved_texts[r.index] for r in reranked.results]
        return final_chunks
    
    return retrieved_texts

print("Retrieval function ready!")

## Step 6: Answer Generation with Cohere Chat

Finally, we take the retrieved chunks and send them to Cohere's chat API along with the user's question. The model uses the chunks as context to generate an accurate answer.

We use `command-r-plus` which is Cohere's best model for RAG tasks. Setting `temperature=0.3` keeps answers more factual and less creative.

In [ ]:
def generate_answer(query, retrieved_chunks, co_client):
    """
    Generate an answer using Cohere's chat API with retrieved documents as context.
    """
    # format chunks as documents for Cohere
    documents = []
    for i, chunk in enumerate(retrieved_chunks):
        documents.append({
            "id": str(i),
            "snippet": chunk,
            "title": f"Chunk {i+1}"
        })
    
    if not documents:
        return "Sorry, couldn't find relevant information in the document."
    
    # call Cohere chat
    response = co_client.chat(
        message=query,
        model="command-r-plus",
        documents=documents,
        temperature=0.3
    )
    
    return response.text

print("Answer generation function ready!")

## Step 7: Complete VectorStore Class

Here's everything wrapped into the `VectorStore` class that our Streamlit app uses. This is the same code as in `src/vectorstore.py`.

In [ ]:
class VectorStore:
    """
    Complete RAG pipeline:
    PDF -> chunks -> embeddings -> Pinecone index -> retrieval
    """
    def __init__(self, pdf_path, cohere_api_key, pinecone_api_key):
        self.pdf_path = pdf_path
        self.co = cohere.Client(cohere_api_key)
        self.pinecone_api_key = pinecone_api_key
        self.chunks = []
        self.embeddings = []
        self.retrieve_top_k = 10
        self.rerank_top_k = 3
        
        # run pipeline
        print("Loading PDF...")
        self.pdf_text = extract_text_from_pdf(pdf_path)
        print(f"Extracted {len(self.pdf_text)} characters")
        
        print("Splitting into chunks...")
        self.chunks = split_into_chunks(self.pdf_text)
        print(f"Created {len(self.chunks)} chunks")
        
        print("Creating embeddings...")
        self.embeddings = embed_texts(self.chunks)
        print(f"Created {len(self.embeddings)} embeddings")
        
        print("Indexing in Pinecone...")
        self.index, self.index_name = create_pinecone_index(pinecone_api_key)
        upsert_vectors(self.index, self.chunks, self.embeddings)
        print("Done! Ready for questions.")
    
    def retrieve(self, query):
        return retrieve_relevant_chunks(
            query, self.index, self.co,
            self.retrieve_top_k, self.rerank_top_k
        )
    
    def cleanup(self):
        try:
            pc = Pinecone(api_key=self.pinecone_api_key)
            pc.delete_index(self.index_name)
            print(f"Cleaned up index: {self.index_name}")
        except:
            pass

print("VectorStore class defined!")

## Step 8: The Chatbot Class

This wraps the retrieval + generation into a simple `respond()` method. Same as `src/chatbot.py`.

In [ ]:
class Chatbot:
    def __init__(self, vectorstore, cohere_api_key):
        self.vectorstore = vectorstore
        self.co = cohere.Client(cohere_api_key)
        self.conversation_id = str(uuid.uuid4())
    
    def respond(self, user_message):
        # retrieve relevant chunks
        chunks = self.vectorstore.retrieve(user_message)
        # generate answer
        answer = generate_answer(user_message, chunks, self.co)
        return answer

print("Chatbot class defined!")

## Step 9: Demo Usage

Here's how you'd use everything together. You need your API keys and a PDF.

**Uncomment below and fill in your keys to try it!**

In [ ]:
# === UNCOMMENT AND FILL IN YOUR DETAILS TO RUN ===

# COHERE_KEY = "your-cohere-api-key"
# PINECONE_KEY = "your-pinecone-api-key"
# PDF_PATH = "your_document.pdf"

# # create vector store (processes the PDF)
# vs = VectorStore(PDF_PATH, COHERE_KEY, PINECONE_KEY)

# # create chatbot
# bot = Chatbot(vs, COHERE_KEY)

# # ask questions
# question = "What is this document about?"
# answer = bot.respond(question)
# print(f"Q: {question}")
# print(f"A: {answer}")

# # cleanup when done
# vs.cleanup()

print("Demo code ready - uncomment and add your API keys to run!")

## Running the Streamlit App

The full interactive app is in `src/app.py`. To run it:

```bash
cd src
streamlit run app.py
```

The app gives you:
- Sidebar for entering API keys
- PDF file upload
- Chat interface for asking questions
- Chat history

## Summary

In this project I built a RAG system that:

1. **Extracts** text from PDF documents using PyMuPDF
2. **Chunks** the text into manageable pieces (~1000 chars each)
3. **Embeds** chunks using Cohere's embed-english-v3.0 model
4. **Indexes** the embeddings in Pinecone for fast similarity search
5. **Retrieves** relevant chunks when a question is asked, then reranks them
6. **Generates** answers using Cohere's command-r-plus model with the retrieved context

### Key Learnings
- RAG helps LLMs answer questions about specific documents they haven't been trained on
- Vector embeddings capture the semantic meaning of text
- Reranking after initial retrieval improves answer quality a lot
- Using the right `input_type` in Cohere embeddings matters

### References
- [Cohere RAG Docs](https://docs.cohere.com/docs/retrieval-augmented-generation-rag)
- [Pinecone Quickstart](https://docs.pinecone.io/guides/get-started/quickstart)
- [Reference Project](https://github.com/VivekChauhan05/RAG_Document_Question_Answering)